In [ ]:
import sys

print("Python version", sys.version)
print("Let's start the proj")

In [ ]:


import librosa
print("Librosa version", librosa.__version__)

import numpy as np
print("Numpy version", np.__version__)

import matplotlib.pyplot as plt


import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
for dirname, _, filenames in os.walk("/kaggle/input"):
    print("Folder:", dirname)

    for filename in filenames[:5]:
        print("  ", filename)

In [ ]:
# CELL 4 — Load and inspect one audio file

import os
import librosa

DATASET_PATH = "/kaggle/input/datasets/yashdogra/speech-commands"

audio_path = os.path.join(DATASET_PATH, "yes", "4c6944d6_nohash_3.wav")

audio, sample_rate = librosa.load(audio_path, sr=None)

print("Audio path:", audio_path)
print("Audio shape:", audio.shape)
print("Sample rate:", sample_rate)
print("Duration:", len(audio) / sample_rate, "seconds")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize = (12, 6))

plt.plot(audio)

plt.title("Waveform of the word: Yes")
plt.xlabel("Sample")
plt.ylabel("Amplitute")

plt.show()

In [ ]:
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

mel_spectrogram = librosa.feature.melspectrogram(
    y = audio,
    sr = sample_rate,
    n_mels = 64,
    n_fft = 1024,
    hop_length = 512
)

mel_db = librosa.power_to_db(
    mel_spectrogram,
    ref = np.max
)

print("Mel-spectrogram shape: ", mel_spectrogram.shape)

In [ ]:
plt.figure(figsize = (12, 5))

librosa.display.specshow(
    mel_db,
    sr=sample_rate,
    hop_length=512,
    x_axis="time",
    y_axis="mel"
)

plt.colorbar(format = "%+2.0f dB")
plt.title("Mel spectrofram of word Yes")
plt.show()

In [ ]:

COMMANDS = [
    "yes",
    "no",
    "up",
    "down",
    "left",
    "right",
    "on",
    "off",
    "stop",
    "go"
]

print("Number of Classes: ", len(COMMANDS))
print("Commands: ", COMMANDS)

print("\nNumber of audio files: ")

for command in COMMANDS:
    command_path = os.path.join(DATASET_PATH, command)

    files = os.listdir(command_path)

    wav_files = [file for file in files if file.endswith(".wav")]

    print(command, ":", len(wav_files))


In [22]:
VALIDATION_FILE = os.path.join(DATASET_PATH, "validation_list.txt")

TEST_FILE = os.path.join(DATASET_PATH, "testing_list.txt")

with open(VALIDATION_FILE, "r") as file:
    validation_files = file.read().splitlines()

with open(TEST_FILE, "r") as file:
    test_files = file.read().splitlines()


print("Validation files: ", len(validation_files))
print("Test files: ", len(test_files))

print("\nFirst 5 validation files: ")
print(validation_files[:5])

print("\nFirst 5 test files: ")
print(test_files[:5])


Validation files:  9981
Test files:  11005

First 5 validation files: 
['right/a69b9b3e_nohash_0.wav', 'right/439c84f4_nohash_1.wav', 'right/409c962a_nohash_1.wav', 'right/dbaf8fc6_nohash_2.wav', 'right/a6d586b7_nohash_1.wav']

First 5 test files: 
['right/bb05582b_nohash_3.wav', 'right/97f4c236_nohash_2.wav', 'right/f2e59fea_nohash_3.wav', 'right/fdb5155e_nohash_2.wav', 'right/dc75148d_nohash_0.wav']


In [4]:
def audio_to_mel(audio_path):
    audio, sample_rate = librosa.load(
        audio_path,
        sr = 16000
    )

    mel_spectrogram = librosa.feature.melspectrogram(
        y = audio,
        sr = sample_rate,
        n_mels = 64,
        n_fft = 1024,
        hop_length = 512
    )

    mel_db = librosa.power_to_db(
        mel_spectrogram,
        ref = np.max
    )

    return mel_db

In [5]:
test_audio_path = os.path.join(
    DATASET_PATH,
    "yes",
    "4c6944d6_nohash_3.wav"
)

mel = audio_to_mel(test_audio_path)

print("Mel-spectrogram shape: ", mel.shape)

Mel-spectrogram shape:  (64, 32)


In [3]:
label_to_index = {
    command: index
    for index, command in enumerate(COMMANDS)
}

index_to_label = {
    index: command
    for command, index in label_to_index.items()
}

print("Label to index: ")
print(label_to_index)

print("Index to label")
print(index_to_label)

Label to index: 
{'yes': 0, 'no': 1, 'up': 2, 'down': 3, 'left': 4, 'right': 5, 'on': 6, 'off': 7, 'stop': 8, 'go': 9}
Index to label
{0: 'yes', 1: 'no', 2: 'up', 3: 'down', 4: 'left', 5: 'right', 6: 'on', 7: 'off', 8: 'stop', 9: 'go'}


In [ ]:
import torch
from torch.utils.data import Dataset

class SpeechCommandsDataset(Dataset):

    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, index):

        audio_path = self.file_paths[index]
        label = self.labels[index]

        mel = audio_to_mel(audio_path)

        mel = torch.tensor(mel, dtype = torch.float32)

        label = torch.tensor(label, dtype = torch.long)

        return mel, label

In [ ]:
# How to make Ai voice sound more humane with intentional noise, voice pull,
# ummmm, and other stuff
# add those noises randomly and trained by dataset

# Folder: /kaggle/input
# Folder: /kaggle/input/datasets
# Folder: /kaggle/input/datasets/yashdogra
# Folder: /kaggle/input/datasets/yashdogra/speech-commands
#    LICENSE
#    testing_list.txt
#    README.md
#    validation_list.txt
# Folder: /kaggle/input/datasets/yashdogra/speech-commands/no
#    4c6944d6_nohash_3.wav
#    4e99c1b7_nohash_3.wav
#    97f4c236_nohash_3.wav
#    cb2929ce_nohash_3.wav
#    ca4d5368_nohash_3.wav